# 🥭 CS22032 Essentials of AI - Custom Mobile Photos Training Notebook
**Module Code**: CS22032 Essentials of Artificial Intelligence  
**Topic**: AI-Based Intelligent Mango Quality & Ripeness Assessment System  
**Platform**: Google Colab Cloud GPU Acceleration (NVIDIA T4)  

--- 
### 📋 Overview
This notebook allows you to upload **real photos of mangoes taken from your mobile phone** and train a 3-block Convolutional Neural Network (`MangoCNN`) in PyTorch.

#### Quality Classes:
- **`Grade_A_Ripe`**: Fresh / Yellow Ripe Mangoes 🥭
- **`Grade_B_Unripe`**: Green / Immature Mangoes 🍏
- **`Grade_C_Overripe`**: Overripe / Bruised / Defective Mangoes 🍂

### ⚡ Step 1: Verify Google Colab GPU Hardware Acceleration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2
import os
import zipfile
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Connected to Google Colab Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Accelerator Name: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not enabled. Go to Runtime -> Change runtime type -> Select T4 GPU")

### 📁 Step 2: Upload Your Mobile Photos ZIP File
Upload your `my_mango_dataset.zip` file containing subfolders (`Grade_A_Ripe`, `Grade_B_Unripe`, `Grade_C_Overripe`).

In [ ]:
print("📥 Click below to upload 'my_mango_dataset.zip' from your computer:")
uploaded = files.upload()

# Extract uploaded zip file
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('dataset')
        print(f"✅ Successfully unzipped {filename} into 'dataset/' folder!")

### 🧠 Step 3: Define PyTorch MangoCNN Neural Network Architecture

In [ ]:
class MangoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(MangoCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc1 = nn.Linear(128 * 4 * 4, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

CLASS_NAMES = ['Grade_A_Ripe', 'Grade_B_Unripe', 'Grade_C_Overripe']
model = MangoCNN(num_classes=3).to(device)
print(model)

### ⚡ Step 4: Train PyTorch Model on GPU for 10 Epochs

In [ ]:
class CustomMangoDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for idx, class_name in enumerate(CLASS_NAMES):
            # Search for class subfolder recursively
            for root, _, files in os.walk(root_dir):
                if class_name in root:
                    for f in files:
                        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                            self.samples.append((os.path.join(root, f), idx))
                            
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = CustomMangoDataset('dataset', transform=transform_train)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"🚀 Training on {len(dataset)} mobile photos for 10 Epochs...")
for epoch in range(1, 11):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outs, 1)
        total += lbls.size(0)
        correct += (preds == lbls).sum().item()
    acc = (correct / total) * 100 if total > 0 else 0
    print(f"Epoch [{epoch:02d}/10] - Loss: {running_loss/max(1,total):.4f} - Accuracy: {acc:.2f}%")

# Save Model Weights
torch.save(model.state_dict(), 'mango_model.pth')
print("🎉 Custom Mobile Model Trained and Saved as 'mango_model.pth'!")

### 📥 Step 5: Download Model File to Local Machine

In [ ]:
files.download('mango_model.pth')
print("📥 Downloading 'mango_model.pth' to your computer...")